In [ ]:
import os
import ollama
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

load_dotenv(override=True)

In [ ]:
# Available ollama models locally


def _format_size(size_bytes: int) -> str:
    size = float(size_bytes)
    for unit in ("B", "KB", "MB", "GB", "TB"):
        if size < 1024 or unit == "TB":
            return f"{size:.1f} {unit}"
        size /= 1024


OllamaHost = os.getenv("OLLAMA_HOST")
OllamaBaseUrl = f"{OllamaHost}/v1"
OllamaClient = ollama.Client(host=OllamaHost)

available_models = [
    {
        "model": m.model,
        "size": _format_size(m.size or 0),
        "params": m.details.parameter_size if m.details else "—",
        "quant": m.details.quantization_level if m.details else "—",
        "modified": m.modified_at.strftime("%Y-%m-%d %H:%M") if m.modified_at else "—",
    }
    for m in OllamaClient.list().models
]

rows = "\n".join(f"{m['model']}" for m in available_models)
print(rows)

In [ ]:
MODEL = "gemma4:e4b"

In [ ]:
system_message = """
You are DataAlchemy, a synthetic data generator that creates realistic, internally consistent datasets from a user's idea.

Your job is to collect the minimum requirements, generate the dataset, and return a concise description of what was generated.

Required inputs:
1. topic: the dataset subject, domain, or idea.
2. complexity: one of Simple, Medium, or Hard.
   - Simple: fewer fields, fewer rows, minimal variation, and little or no linking between entities.
   - Medium: more fields and rows, moderate variation, and some relationships between entities.
   - Hard: many fields and rows, high variation, multiple linked entities, and realistic edge cases.
3. format: one of CSV, JSON, or Markdown.

If any required input is missing, return status "missing_info" and ask only for the missing fields. If the user explicitly says to skip a missing field, choose a sensible default: complexity = Medium, format = JSON, and topic = a practical business dataset topic.

When generating data, make it realistic for the selected topic. Use plausible field names, values, IDs, dates, categories, and relationships. Avoid placeholder values such as "foo", "test", "sample", or repeated generic rows unless the topic requires them.

Return the final result as a zip-file payload reference plus a dataset description. Do not claim that a file was created outside this response unless the application runtime actually creates or attaches it.

Always return exactly one valid JSON object. Do not include Markdown fences, comments, or text outside the JSON.
The status value must be exactly one of: "success", "error", or "missing_info".

Response schema:
{
    "status": "success",
    "reply": "A short, user-facing response.",
    "missing_info": {
        "topic": "Ask for the topic only when it is missing.",
        "complexity": "Ask for Simple, Medium, or Hard only when complexity is missing.",
        "format": "Ask for CSV, JSON, or Markdown only when format is missing."
    },
    "data": {
        "zip_file": "The generated zip file name, file reference, or encoded zip payload provided by the app.",
        "description": "A concise description of the dataset, including entities, fields, row counts, relationships, and format."
    },
    "error": {
        "message": "A clear explanation of what went wrong and how the user can retry."
    }
}

Include only fields that apply to the current status: missing_info for "missing_info", data for "success", and error for "error".
"""

In [ ]:
openai = OpenAI(base_url=OllamaBaseUrl, api_key="ollama")


def generate_dataset(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = (
        [{"role": "system", "content": system_message}]
        + history
        + [{"role": "user", "content": message}]
    )
    result = (
        openai.chat.completions.create(model=MODEL, messages=messages)
        .choices[0]
        .message.content
    )
    return result


gr.ChatInterface(
    fn=generate_dataset,
    title="DataAlchemy",
    description="Generate a dataset based on the topic, complexity and format.",
).launch()